# Export & Vectorization — Monitor do Fogo (Brasil)

Pipeline de 4 etapas para exportar, mosaicar e vetorizar os mapas mensais
de area queimada do Monitor do Fogo no Brasil.

| Etapa | O que faz |
|-------|-----------|
| **Export** | Envia as imagens do GEE para tiles `.tif` no GCS (`temp/`) |
| **Mosaico** | Junta os tiles em um unico COG por mes (`monthly_burned/`) |
| **Vetorizacao** | Converte o raster em shapefile com `unique_id` (`monthly_vectors/`) |
| **Upload GEE** | Publica o shapefile como FeatureCollection no Earth Engine |

---

### Como usar (em ordem)

1. Execute as celulas abaixo **uma por vez**, de cima para baixo.
2. A celula 5 abre a interface. Clique em **Sincronizar** para ver o status.
3. Marque os meses pendentes e execute as celulas 6 a 9 em sequencia.
4. A etapa de **Export** dispara tarefas no GEE — aguarde a conclusao delas antes de prosseguir.
5. Apos cada etapa, clique **Sincronizar** para atualizar a grid.

In [ ]:
# Instala as dependencias necessarias (GDAL, Earth Engine, GCS, etc.)
!apt-get update -q
!apt-get install -y gdal-bin python3-gdal
!pip install --quiet gcsfs earthengine-api geopandas rasterio psutil ipywidgets

print('Dependencias instaladas.')

In [ ]:
# Clona o repositorio com os scripts do pipeline
!rm -rf /content/brazil-fire
!git clone --branch main https://github.com/mapbiomas/brazil-fire.git /content/brazil-fire

import sys
sys.path.append('/content/brazil-fire/mapbiomas_fire_monitor/version_01')

print('Repositorio clonado e path configurado.')

In [ ]:
# Edite as 3 variaveis abaixo conforme necessario.
GEE_PROJECT  = "ee-ipam"
BUCKET_GCS   = "mapbiomas-fire"
EXPORT_FLAG  = "WS_fire_monitor-"

from google.colab import auth
auth.authenticate_user()

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

import export_and_vectorization.state as _state
_state.GEE_PROJECT = GEE_PROJECT
_state.BUCKET = BUCKET_GCS

import export_and_vectorization.export as _export
_export.EXPORT_FLAG = EXPORT_FLAG

print('Autenticacao concluida. Projeto GEE:', GEE_PROJECT, '| Bucket:', BUCKET_GCS)

In [ ]:
# LIMPEZA — apaga tiles e mosaicos >= 2025 do GCS para reprocessamento.
# Execute esta celula ANTES de abrir a UI / clicar em Sincronizar.
# Rode-a apenas se precisar refazer exports/mosaicos dos ultimos meses.

from export_and_vectorization.state import _get_fs, BUCKET, TILES_PREFIX, MOSAIC_PREFIX

fs = _get_fs()
CUTOFF = 2025

print('Procurando arquivos >= {} ...'.format(CUTOFF))

tiles = fs.glob(f"{BUCKET}/{TILES_PREFIX}/fire_monitor_v1_monthly_burned_brazil_*.tif")
for t in tiles:
    y = t.split('/')[-1].replace('fire_monitor_v1_monthly_burned_brazil_', '').split('_')[0]
    if len(y) == 4 and int(y) >= CUTOFF:
        fs.rm(t)
        print(f'  DEL tiles: {t}')

mosaics = fs.glob(f"{BUCKET}/{MOSAIC_PREFIX}/monthly_burned-brazil_*.tif")
for m in mosaics:
    y = m.split('/')[-1].replace('monthly_burned-brazil_', '').split('_')[0]
    if len(y) == 4 and int(y) >= CUTOFF:
        fs.rm(m)
        print(f'  DEL mosaic: {m}')

print('Limpeza concluida. Agora rode a celula da UI e clique Sincronizar.')


In [ ]:
# TESTE — verifica que image.geometry().bounds() esta no Brasil.
# O bounding box deve estar aproximadamente entre:
#   lon: -74 a -34
#   lat: -33 a 5
# Se as coordenadas estiverem nessa faixa, a correcao do export funcionou.

from export_and_vectorization.export import get_image_for_month

test_months = [(2025, 1), (2025, 6), (2026, 1)]
for y, m in test_months:
    img = get_image_for_month(y, m)
    if img is None:
        print(f'{y}_{m:02d}: sem imagem na colecao')
        continue
    bounds = img.geometry().bounds().getInfo()
    coords = bounds['coordinates'][0]
    print(f"{y}_{m:02d}: lon={coords[0][0]:.2f}..{coords[2][0]:.2f}  lat={coords[0][1]:.2f}..{coords[1][1]:.2f}")


In [ ]:
# Abre a interface com a grid de meses.
# A grid ja mostra os meses disponiveis na colecao.
# Clique em SINCRONIZAR para verificar o status de cada etapa.
from export_and_vectorization.ui import run_ui

ui = run_ui()

In [ ]:
# ETAPA 1: Export — envia imagens do GEE para tiles .tif no GCS.
# Esta etapa dispara tarefas no Earth Engine e pode demorar.
# Acompanhe o progresso no GEE: https://code.earthengine.google.com/tasks
# Apos a conclusao, volte na celula 5 e clique em Sincronizar.
from export_and_vectorization.export import export_selected

export_selected(ui, logger=ui._log)

In [ ]:
# ETAPA 2: Mosaico — junta os tiles em um unico COG por mes.
# Execucao paralela (usa todos os nucleos da CPU disponiveis).
# So processa meses que ja tem tiles no GCS.
from export_and_vectorization.mosaic import mosaic_selected

mosaic_selected(ui, logger=ui._log)

In [ ]:
# ETAPA 3: Vetorizacao — converte cada mosaico raster em shapefile.
# Usa gdal_polygonize (ignora pixel=0) e adiciona coluna unique_id.
# Execucao paralela (usa todos os nucleos da CPU disponiveis).
# So processa meses que ja tem mosaico no GCS.
from export_and_vectorization.vectorize import vectorize_selected

vectorize_selected(ui, logger=ui._log)

In [ ]:
# ETAPA 4: Upload GEE — publica os shapefiles como FeatureCollection.
# Execucao sequencial (earthengine upload table).
# So processa meses que ja tem shapefile no GCS.
from export_and_vectorization.vectorize import gee_upload_selected

gee_upload_selected(ui, logger=ui._log)